# Task 4 - Data Preparation

In [1]:
import pandas as pd
from math import ceil
from datetime import timedelta

## Read personal information about customers

In [2]:
personal_info = pd.read_csv('Data/personal_info.csv')
personal_info

,id,birth_date,gender,subscription_start,subscription_end
0,0,1986-10-30,F,2010-08-18,2018-01-01
1,1,1967-03-22,M,1997-07-05,2018-01-01
2,2,1965-12-25,M,2003-05-23,2018-01-01
3,3,1988-06-17,F,2013-09-14,2018-01-01
4,4,1964-12-18,M,2004-01-28,2018-01-01
...,...,...,...,...,...
995,995,1962-08-11,M,2003-05-31,2018-01-01
996,996,1987-12-10,F,2007-11-18,2018-01-01
997,997,1989-10-24,F,2007-10-24,2017-12-13
998,998,1980-04-04,F,2013-10-03,2018-01-01


## Read data sent by customers

In [3]:
data = pd.read_csv('Data/data.csv')
data.drop('Unnamed: 0', axis=True, inplace=True)
data

,id,date,size,operator,roaming
0,0,02.01.2015,443,21,1
1,0,03.01.2015,1,0,0
2,0,04.01.2015,21,0,0
3,0,04.01.2015,286,11,1
4,0,05.01.2015,333,19,1
...,...,...,...,...,...
740172,999,25.12.2017,35,11,1
740173,999,27.12.2017,12,0,0
740174,999,28.12.2017,19,0,0
740175,999,29.12.2017,7,7,1


## Read invoices

In [4]:
invoice = pd.read_csv('Data/invoice.csv')
invoice.drop('Unnamed: 0', axis=True, inplace=True)
invoice

,id,date,amount
0,0,1.2015,109
1,0,1.2016,24
2,0,1.2017,60
3,0,2.2015,61
4,0,2.2016,25
...,...,...,...
35995,999,11.2016,65
35996,999,11.2017,59
35997,999,12.2015,52
35998,999,12.2016,62


## Read phone calls

In [5]:
phone_calls = pd.read_csv('Data/phone_calls.csv')
phone_calls.drop('Unnamed: 0', axis=True, inplace=True)
phone_calls

,id,date,duration,operator,roaming
0,0,01.01.2015,49,1,0
1,0,01.01.2015,286,2,0
2,0,02.01.2015,899,1,0
3,0,02.01.2015,92,4,0
4,0,02.01.2015,831,12,1
...,...,...,...,...,...
1087464,999,29.12.2017,12,4,0
1087465,999,30.12.2017,525,0,0
1087466,999,30.12.2017,6,1,0
1087467,999,30.12.2017,16,3,0


## Read sms

In [6]:
sms = pd.read_csv('Data/sms.csv')
sms.drop('Unnamed: 0', axis=True, inplace=True)
sms

,id,date,num,operator,roaming
0,0,01.01.2015,7,2,0
1,0,02.01.2015,4,0,0
2,0,02.01.2015,1,2,0
3,0,02.01.2015,13,23,1
4,0,03.01.2015,1,0,0
...,...,...,...,...,...
939544,999,26.12.2017,1,0,0
939545,999,26.12.2017,2,22,1
939546,999,27.12.2017,2,3,0
939547,999,31.12.2017,4,9,1


## Create final dataframe

In [7]:
def merge_group(df_first: pd.DataFrame, df_second: pd.DataFrame, format):
    merged_data = pd.merge(df_first, df_second, on=['id', 'id'])
    merged_data['date'] = pd.to_datetime(merged_data['date'], format=format)
    merged_data['subscription_end'] = pd.to_datetime(merged_data['subscription_end'], format='%Y.%m.%d')
    merged_data = merged_data[merged_data['date'] < (merged_data['subscription_end'] - timedelta(days=29))]
    return merged_data.groupby('id')

In [8]:
average_data = merge_group(data, personal_info, '%d.%m.%Y')
avg_data = average_data['size'].mean()
avg_data_roaming = average_data['roaming'].mean()
del average_data

average_calls = merge_group(phone_calls, personal_info, '%d.%m.%Y')
avg_duration = average_calls['duration'].mean()
avg_calls_roaming = average_calls['roaming'].mean()
del average_calls

average_sms = merge_group(sms, personal_info, '%d.%m.%Y')
avg_sms = average_sms['num'].mean()
avg_sms_roaming = average_sms['roaming'].mean()
del average_sms

new_dates = []
for date in invoice['date']:
    new_dates.append(str(date))
    
invoice['date'] = new_dates
average_invoice = merge_group(invoice, personal_info, '%m.%Y')
avg_invoice = average_invoice['amount'].mean()
total_invoices = average_invoice['amount'].sum()
del average_invoice

In [9]:
customers = personal_info.copy()
customers['avg_invoice'] = avg_invoice
customers['total_invoices'] = total_invoices
customers['sms'] = avg_sms
customers['sms_roam'] = avg_sms_roaming
customers['data'] = avg_data
customers['data_roam'] = avg_data_roaming
customers['phone_calls'] = avg_duration
customers['calls_roam'] = avg_calls_roaming
customers['churn'] = customers['subscription_end'] < '2017-12-20'
customers.set_index('id')

,birth_date,gender,subscription_start,subscription_end,avg_invoice,total_invoices,sms,sms_roam,data,data_roam,phone_calls,calls_roam,churn
id,,,,,,,,,,,,,
0,1986-10-30,F,2010-08-18,2018-01-01,49.805556,1793.0,2.508103,0.209724,128.034166,0.637319,315.846672,0.096040,False
1,1967-03-22,M,1997-07-05,2018-01-01,43.694444,1573.0,2.515474,0.184720,128.798684,0.667105,305.376511,0.087219,False
2,1965-12-25,M,2003-05-23,2018-01-01,49.805556,1793.0,2.544343,0.209990,127.206107,0.679389,315.222679,0.112572,False
3,1988-06-17,F,2013-09-14,2018-01-01,44.083333,1587.0,2.519157,0.208812,130.845477,0.688442,297.879414,0.093023,False
4,1964-12-18,M,2004-01-28,2018-01-01,45.500000,1638.0,2.574512,0.204522,132.716000,0.684000,301.180602,0.091973,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,1962-08-11,M,2003-05-31,2018-01-01,43.194444,1555.0,2.569418,0.220450,127.200000,0.673418,296.001727,0.081174,False
996,1987-12-10,F,2007-11-18,2018-01-01,43.777778,1576.0,2.568119,0.230769,147.087224,0.687961,305.024076,0.082545,False
997,1989-10-24,F,2007-10-24,2017-12-13,42.371429,1483.0,2.503945,0.208087,133.477593,0.668374,208.091509,0.106348,True


In [10]:
# Add 'age' column in dataframe
customers['birth_date'] = pd.to_datetime(customers['birth_date'])
ages = [ceil(age.days / 365) for age in pd.to_datetime('2018-01-01') - customers['birth_date']]
customers.insert(2, 'age', ages)

In [11]:
# Calculate subscription length
customers['subscription_start'] = pd.to_datetime(customers['subscription_start'])
customers['subscription_end'] = pd.to_datetime(customers['subscription_end'])
subscription_lenght = [days.days for days in customers['subscription_end'] - customers['subscription_start']]
customers.insert(6, 'subscription_length', subscription_lenght)

In [12]:
customers

,id,birth_date,age,gender,subscription_start,subscription_end,subscription_length,avg_invoice,total_invoices,sms,sms_roam,data,data_roam,phone_calls,calls_roam,churn
0,0,1986-10-30,32,F,2010-08-18,2018-01-01,2693,49.805556,1793.0,2.508103,0.209724,128.034166,0.637319,315.846672,0.096040,False
1,1,1967-03-22,51,M,1997-07-05,2018-01-01,7485,43.694444,1573.0,2.515474,0.184720,128.798684,0.667105,305.376511,0.087219,False
2,2,1965-12-25,53,M,2003-05-23,2018-01-01,5337,49.805556,1793.0,2.544343,0.209990,127.206107,0.679389,315.222679,0.112572,False
3,3,1988-06-17,30,F,2013-09-14,2018-01-01,1570,44.083333,1587.0,2.519157,0.208812,130.845477,0.688442,297.879414,0.093023,False
4,4,1964-12-18,54,M,2004-01-28,2018-01-01,5087,45.500000,1638.0,2.574512,0.204522,132.716000,0.684000,301.180602,0.091973,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
995,995,1962-08-11,56,M,2003-05-31,2018-01-01,5329,43.194444,1555.0,2.569418,0.220450,127.200000,0.673418,296.001727,0.081174,False
996,996,1987-12-10,31,F,2007-11-18,2018-01-01,3697,43.777778,1576.0,2.568119,0.230769,147.087224,0.687961,305.024076,0.082545,False
997,997,1989-10-24,29,F,2007-10-24,2017-12-13,3703,42.371429,1483.0,2.503945,0.208087,133.477593,0.668374,208.091509,0.106348,True
998,998,1980-04-04,38,F,2013-10-03,2018-01-01,1551,48.638889,1751.0,2.696824,0.201155,143.787421,0.656604,298.433913,0.101739,False


In [13]:
customers.to_csv('Data/train_data_29.csv', index=False)